In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [2]:
import sys
print(sys.executable)

/Users/rohit/venv/time-series-venv/bin/python


In [3]:
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)

In [4]:
import os
os.getcwd()

'/Users/rohit/Adaptive-Time-Series-Forecasting-Engine/src/models/classical'

In [6]:
df = pd.read_csv("../../../data/raw/sales_train_validation.csv")

In [7]:
df

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,FOODS_3_823_WI_3_validation,FOODS_3_823,FOODS_3,FOODS,WI_3,WI,0,0,2,2,...,2,0,0,0,0,0,1,0,0,1
30486,FOODS_3_824_WI_3_validation,FOODS_3_824,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
30487,FOODS_3_825_WI_3_validation,FOODS_3_825,FOODS_3,FOODS,WI_3,WI,0,6,0,2,...,2,1,0,2,0,1,0,0,1,0
30488,FOODS_3_826_WI_3_validation,FOODS_3_826,FOODS_3,FOODS,WI_3,WI,0,0,0,0,...,0,0,1,0,0,1,0,3,1,3


In [8]:
item_id = df.iloc[0]['id']

In [10]:
item_id

'HOBBIES_1_001_CA_1_validation'

In [11]:
sales_cols = [col for col in df.columns if col.startswith('d_')]

In [15]:
sales_cols

['d_1',
 'd_2',
 'd_3',
 'd_4',
 'd_5',
 'd_6',
 'd_7',
 'd_8',
 'd_9',
 'd_10',
 'd_11',
 'd_12',
 'd_13',
 'd_14',
 'd_15',
 'd_16',
 'd_17',
 'd_18',
 'd_19',
 'd_20',
 'd_21',
 'd_22',
 'd_23',
 'd_24',
 'd_25',
 'd_26',
 'd_27',
 'd_28',
 'd_29',
 'd_30',
 'd_31',
 'd_32',
 'd_33',
 'd_34',
 'd_35',
 'd_36',
 'd_37',
 'd_38',
 'd_39',
 'd_40',
 'd_41',
 'd_42',
 'd_43',
 'd_44',
 'd_45',
 'd_46',
 'd_47',
 'd_48',
 'd_49',
 'd_50',
 'd_51',
 'd_52',
 'd_53',
 'd_54',
 'd_55',
 'd_56',
 'd_57',
 'd_58',
 'd_59',
 'd_60',
 'd_61',
 'd_62',
 'd_63',
 'd_64',
 'd_65',
 'd_66',
 'd_67',
 'd_68',
 'd_69',
 'd_70',
 'd_71',
 'd_72',
 'd_73',
 'd_74',
 'd_75',
 'd_76',
 'd_77',
 'd_78',
 'd_79',
 'd_80',
 'd_81',
 'd_82',
 'd_83',
 'd_84',
 'd_85',
 'd_86',
 'd_87',
 'd_88',
 'd_89',
 'd_90',
 'd_91',
 'd_92',
 'd_93',
 'd_94',
 'd_95',
 'd_96',
 'd_97',
 'd_98',
 'd_99',
 'd_100',
 'd_101',
 'd_102',
 'd_103',
 'd_104',
 'd_105',
 'd_106',
 'd_107',
 'd_108',
 'd_109',
 'd_110',
 'd_111'

In [27]:
item_sales = df[df['id'] == item_id][sales_cols].T

In [28]:
item_sales

,0
d_1,0
d_2,0
d_3,0
d_4,0
d_5,0
...,...
d_1909,1
d_1910,3
d_1911,0
d_1912,1


In [29]:
item_sales.columns = ['sales']

In [30]:
item_sales

,sales
d_1,0
d_2,0
d_3,0
d_4,0
d_5,0
...,...
d_1909,1
d_1910,3
d_1911,0
d_1912,1


In [32]:
item_sales.index = pd.date_range(start='2011-01-29', periods=len(item_sales), freq='D')

In [33]:
item_sales

,sales
2011-01-29,0
2011-01-30,0
2011-01-31,0
2011-02-01,0
2011-02-02,0
...,...
2016-04-20,1
2016-04-21,3
2016-04-22,0
2016-04-23,1


In [34]:
item_sales['sales'] = item_sales['sales'].astype(float)

In [35]:
item_sales

,sales
2011-01-29,0.0
2011-01-30,0.0
2011-01-31,0.0
2011-02-01,0.0
2011-02-02,0.0
...,...
2016-04-20,1.0
2016-04-21,3.0
2016-04-22,0.0
2016-04-23,1.0


In [ ]:
pr

In [39]:
plt.figure()
plt.plot(
    item_sales['Date'],
    item_sales['sales'],
    label='Daily Sales',
    color='blue',
    marker='o'
)

plt.title(f'Daily Sales for {item_id}')
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

KeyError: 'Date'

<Figure size 1200x600 with 0 Axes>